# Wind Power Exploratory Data Analysis

This project performs an independent exploratory data analysis (EDA) of the **Spatial Dynamic Wind Power Forecasting (SDWPF)** dataset.

The dataset contains observations from **134 wind turbines** in a wind farm in China. The main analytical focus is understanding the distributions, relationships, data-quality issues, and temporal behaviour of wind conditions and active power generation.

> **Note:** This notebook is a cleaned and standalone version of a course-based exploration. It does not depend on the course-specific `utils.py` module or interactive widgets.


## 1. Objectives

The analysis focuses on:

1. Loading and inspecting the wind-turbine dataset.
2. Quantifying and handling missing observations.
3. Examining descriptive statistics and data distributions.
4. Comparing turbine-level performance.
5. Investigating the relationship between wind speed and active power.
6. Examining correlations among numerical variables.
7. Creating a datetime variable for temporal analysis.
8. Identifying potential sensor anomalies.
9. Summarizing the main findings and possible next steps toward wind-power prediction.


## 2. Dataset

The SDWPF dataset was provided by Longyuan Power Group and was used in the Baidu KDD Cup 2022. The dataset includes turbine identifiers, time information, wind measurements, temperature measurements, nacelle/blade variables, reactive power, and active power.

`Patv (kW)` is treated as the main output variable for this exploratory analysis and is the variable that would be predicted in a subsequent machine-learning stage.


In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")


In [ ]:
# Locate the dataset
DATA_PATH = Path("data/wtbdata_245days.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Place wtbdata_245days.csv inside the project's data/ folder."
    )

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")


In [ ]:
# Standardize column names
df.columns = [
    "TurbID",
    "Day",
    "Tmstamp",
    "Wspd (m/s)",
    "Wdir (°)",
    "Etmp (°C)",
    "Itmp (°C)",
    "Ndir (°)",
    "Pab1 (°)",
    "Pab2 (°)",
    "Pab3 (°)",
    "Prtv (kW)",
    "Patv (kW)"
]

df.head()


## 3. Initial Dataset Inspection

Before cleaning the data, inspect its dimensions, data types, unique turbines, and basic structure.


In [ ]:
# Basic structure
print("Number of observations:", len(df))
print("Number of variables:", df.shape[1])
print("Number of turbines:", df["TurbID"].nunique())

display(df.info())


In [ ]:
# Unique turbine IDs
turbine_ids = sorted(df["TurbID"].unique())

print(f"Turbine IDs ({len(turbine_ids)}):")
print(turbine_ids)


## 4. Missing-Value Analysis

The original dataset contains missing measurements across the numerical sensor variables. Here, missingness is quantified both by column and by row.

For this exploratory analysis, rows containing missing sensor measurements are removed. This is a simple cleaning choice appropriate for the EDA stage; a future prediction model should evaluate whether interpolation or another missing-data strategy is preferable.


In [ ]:
# Missing values by column
missing = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
)

missing_percent = (missing / len(df) * 100).round(3)

missing_summary = pd.DataFrame({
    "Missing Values": missing,
    "Missing (%)": missing_percent
})

missing_summary


In [ ]:
# Rows containing at least one missing value
rows_with_missing = df.isna().any(axis=1).sum()

print(f"Total rows: {len(df):,}")
print(f"Rows with missing values: {rows_with_missing:,}")
print(f"Percentage of rows with missing values: "
      f"{rows_with_missing / len(df) * 100:.3f}%")


In [ ]:
# Remove rows with missing values
df_clean = df.dropna().copy()

print(f"Rows before cleaning: {len(df):,}")
print(f"Rows after cleaning:  {len(df_clean):,}")
print(f"Rows removed:         {len(df) - len(df_clean):,}")


## 5. Data-Quality Checks

The descriptive statistics in the source analysis show unusually large positive and negative values for several sensor variables, particularly temperature and direction measurements. These values should be treated as potential sensor anomalies rather than automatically interpreted as physical conditions.

The purpose here is to **identify** suspicious observations; they are not automatically deleted unless a clear rule is defined.


In [ ]:
# Numerical variables
numerical_features = [
    "Wspd (m/s)",
    "Wdir (°)",
    "Etmp (°C)",
    "Itmp (°C)",
    "Ndir (°)",
    "Pab1 (°)",
    "Pab2 (°)",
    "Pab3 (°)",
    "Prtv (kW)",
    "Patv (kW)"
]

df_clean[numerical_features].describe().T


In [ ]:
# Inspect extreme temperature observations
temperature_cols = ["Etmp (°C)", "Itmp (°C)"]

temperature_extremes = (
    df_clean[temperature_cols]
    .agg(["min", "max", "mean", "median"])
    .T
)

temperature_extremes


In [ ]:
# Count potentially suspicious temperature values
suspicious_temp = (
    (df_clean["Etmp (°C)"] < -50) |
    (df_clean["Itmp (°C)"] < -50)
)

print("Rows with temperature below -50°C:",
      suspicious_temp.sum())


## 6. Turbine-Level Performance

To compare turbines, calculate the average and total active power for each turbine.

Total active power is used as a simple proxy for cumulative energy output in this exploratory comparison. Because the observations are recorded at fixed time intervals, this provides a useful relative comparison among turbines.


In [ ]:
turbine_summary = (
    df_clean.groupby("TurbID")
    .agg(
        mean_power_kW=("Patv (kW)", "mean"),
        median_power_kW=("Patv (kW)", "median"),
        max_power_kW=("Patv (kW)", "max"),
        total_power_kW=("Patv (kW)", "sum"),
        mean_wind_speed=("Wspd (m/s)", "mean")
    )
    .sort_values("total_power_kW", ascending=False)
)

turbine_summary.head(10)


In [ ]:
# Select the 10 turbines with the highest cumulative active power
top_10_turbines = turbine_summary.head(10).index.tolist()

print("Top 10 turbines:", top_10_turbines)


In [ ]:
# Plot average power of the top 10 turbines
top10_plot = turbine_summary.loc[top_10_turbines].sort_values("mean_power_kW")

plt.figure(figsize=(10, 6))
plt.barh(top10_plot.index.astype(str), top10_plot["mean_power_kW"])
plt.xlabel("Mean Active Power (kW)")
plt.ylabel("Turbine ID")
plt.title("Average Active Power of Top 10 Turbines")
plt.tight_layout()
plt.show()


## 7. Distribution of Numerical Variables

Histograms provide an overview of the distributions of wind speed, power, temperatures, and turbine operating variables.

Because the full dataset contains millions of observations, a sample is used for some visualizations to keep the notebook responsive.


In [ ]:
# Sample observations for visualization
sample_size = min(100_000, len(df_clean))
df_sample = df_clean.sample(sample_size, random_state=42)

print(f"Visualization sample size: {len(df_sample):,}")


In [ ]:
# Histograms for key variables
features_to_plot = [
    "Wspd (m/s)",
    "Etmp (°C)",
    "Itmp (°C)",
    "Prtv (kW)",
    "Patv (kW)"
]

for feature in features_to_plot:
    plt.figure(figsize=(9, 5))
    plt.hist(df_sample[feature], bins=60)
    plt.xlabel(feature)
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {feature}")
    plt.tight_layout()
    plt.show()


## 8. Box Plots

Box plots help identify differences in active-power distributions across turbines and highlight potential outliers.


In [ ]:
# Active-power distributions for the top 10 turbines
plot_data = df_clean[df_clean["TurbID"].isin(top_10_turbines)]

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=plot_data,
    x="TurbID",
    y="Patv (kW)"
)
plt.xlabel("Turbine ID")
plt.ylabel("Active Power (kW)")
plt.title("Active Power Distribution Across Top 10 Turbines")
plt.tight_layout()
plt.show()


## 9. Wind Speed vs. Active Power

A key relationship in wind-energy analysis is the relationship between wind speed and generated power.

The theoretical power curve describes how turbine power should vary with wind speed. The dataset does not provide the theoretical curve directly, so the analysis below focuses on the empirical relationship observed in the data.


In [ ]:
# Scatterplot using a sample of observations
plt.figure(figsize=(10, 6))
plt.scatter(
    df_sample["Wspd (m/s)"],
    df_sample["Patv (kW)"],
    alpha=0.15,
    s=8
)
plt.xlabel("Wind Speed (m/s)")
plt.ylabel("Active Power (kW)")
plt.title("Wind Speed vs. Active Power")
plt.tight_layout()
plt.show()


In [ ]:
# Quantify average power across wind-speed bins
df_clean["WindSpeedBin"] = pd.cut(
    df_clean["Wspd (m/s)"],
    bins=np.arange(0, df_clean["Wspd (m/s)"].max() + 1, 1),
    include_lowest=True
)

wind_speed_power = (
    df_clean.groupby("WindSpeedBin", observed=True)["Patv (kW)"]
    .mean()
)

plt.figure(figsize=(10, 6))
plt.plot(
    wind_speed_power.index.astype(str),
    wind_speed_power.values,
    marker="o"
)
plt.xlabel("Wind Speed Bin (m/s)")
plt.ylabel("Mean Active Power (kW)")
plt.title("Mean Active Power by Wind Speed")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()


## 10. Correlation Analysis

Pearson correlation coefficients are calculated for the numerical variables.

Correlation can identify linear associations, but it does not establish causality and may not capture nonlinear relationships.


In [ ]:
correlation_matrix = df_clean[numerical_features].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    square=True
)
plt.title("Pearson Correlation Matrix")
plt.tight_layout()
plt.show()


In [ ]:
# Variables most strongly correlated with active power
power_correlations = (
    correlation_matrix["Patv (kW)"]
    .drop("Patv (kW)")
    .sort_values(key=lambda x: x.abs(), ascending=False)
)

power_correlations


## 11. Time-Series Preparation

The original data stores the day number and measurement time separately. The first day corresponds to **May 1, 2020**. These fields are combined into a single datetime variable for temporal analysis.


In [ ]:
# Convert day and timestamp into a datetime variable
start_date = pd.Timestamp("2020-05-01")

df_clean["datetime"] = (
    start_date
    + pd.to_timedelta(df_clean["Day"].astype(int) - 1, unit="D")
    + pd.to_timedelta(df_clean["Tmstamp"] + ":00")
)

df_clean[["TurbID", "Day", "Tmstamp", "datetime"]].head()


## 12. Time-Series Analysis

The following plots show how wind speed and active power vary over time for one representative turbine.

A single turbine is used to keep the visualization readable. The turbine ID can be changed to investigate other turbines.


In [ ]:
# Select a turbine for temporal analysis
selected_turbine = top_10_turbines[0]

turbine_ts = (
    df_clean[df_clean["TurbID"] == selected_turbine]
    .sort_values("datetime")
)

print(f"Selected turbine: {selected_turbine}")
print(f"Observations: {len(turbine_ts):,}")


In [ ]:
# Wind speed over time
plt.figure(figsize=(14, 5))
plt.plot(
    turbine_ts["datetime"],
    turbine_ts["Wspd (m/s)"],
    linewidth=0.8
)
plt.xlabel("Date")
plt.ylabel("Wind Speed (m/s)")
plt.title(f"Wind Speed Over Time — Turbine {selected_turbine}")
plt.tight_layout()
plt.show()


In [ ]:
# Active power over time
plt.figure(figsize=(14, 5))
plt.plot(
    turbine_ts["datetime"],
    turbine_ts["Patv (kW)"],
    linewidth=0.8
)
plt.xlabel("Date")
plt.ylabel("Active Power (kW)")
plt.title(f"Active Power Over Time — Turbine {selected_turbine}")
plt.tight_layout()
plt.show()


## 13. Daily Aggregation

Daily averages provide a less noisy view of the relationship between wind conditions and power generation.


In [ ]:
daily_summary = (
    turbine_ts
    .set_index("datetime")
    .resample("D")
    .agg(
        mean_wind_speed=("Wspd (m/s)", "mean"),
        mean_power=("Patv (kW)", "mean"),
        max_power=("Patv (kW)", "max")
    )
)

daily_summary.head()


In [ ]:
# Daily mean wind speed and active power
fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.plot(
    daily_summary.index,
    daily_summary["mean_wind_speed"],
    label="Mean Wind Speed"
)
ax1.set_xlabel("Date")
ax1.set_ylabel("Mean Wind Speed (m/s)")

ax2 = ax1.twinx()
ax2.plot(
    daily_summary.index,
    daily_summary["mean_power"],
    label="Mean Active Power"
)
ax2.set_ylabel("Mean Active Power (kW)")

plt.title(f"Daily Wind Conditions and Power — Turbine {selected_turbine}")
fig.tight_layout()
plt.show()


## 14. Key Findings

The EDA provides the following areas of insight:

- The dataset contains observations from multiple wind turbines over an extended period.
- Missing measurements are concentrated in the numerical sensor variables and represent a relatively small proportion of the full dataset.
- Turbine-level active-power distributions differ, indicating variation in observed operating performance.
- Wind speed and active power show an important empirical relationship that is central to wind-power prediction.
- Correlation analysis provides a first indication of which numerical variables are associated with active power.
- Time-series analysis reveals temporal variation in both wind conditions and generated power.
- Extremely unusual temperature measurements should be investigated as potential sensor errors before they are used in predictive modelling.


## 15. Conclusion

This exploratory analysis establishes a data-quality and statistical foundation for a future **wind-power prediction** project.

The next stage could use the cleaned dataset to develop predictive models using machine-learning methods. Potential extensions include:

- feature engineering from time and weather variables,
- lagged wind-power features,
- turbine-specific models,
- regression models,
- tree-based machine-learning algorithms,
- model evaluation using time-aware train/test splits,
- anomaly detection,
- and comparison of predicted versus observed power curves.


## 16. Data Source

The analysis is based on the **Spatial Dynamic Wind Power Forecasting (SDWPF)** dataset described in:

> Spatial Dynamic Wind Power Forecasting, used in the Baidu KDD Cup 2022.

Original dataset/paper information:
- https://arxiv.org/abs/2208.04360
- https://aistudio.baidu.com/aistudio/competition/detail/152/0/introduction

The dataset is not original data collected by the author of this notebook; the project focuses on independent analysis and Python implementation.
